In [2]:
from ingest import load_faq_data
documents = load_faq_data()

In [4]:
documents[10]

{'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: How many hours per week am I expected to spend on this course?',
 'answer': 'It depends on your background and previous experience with modules. It is expected to require about 5 - 15 hours per week.\n\nYou can also calculate it yourself using [this data](https://github.com/DataTalksClub/zoomcamp-analytics/tree/main/data/de-zoomcamp-2023) and then update this answer.',
 'doc_id': '316180784f'}

In [3]:
documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

len(documents_llm)

113

In [5]:
documents = documents_llm

In [9]:
doc = documents[0]
print(doc["doc_id"])
print(doc["question"])
print(doc["answer"])

74eb249bbf
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.


In [10]:
from pydantic import BaseModel

class Questions(BaseModel):
    questions: list[str]

In [11]:
data_gen_instructions = """
You emulate a student who's taking our course.
Formulate 5 questions this student might ask based on a FAQ record. The record
should contain the answer to the questions, and the questions should be complete and not too short.
If possible, use as fewer words as possible from the record.

The output should resemble how people ask questions
on the internet. Not too formal, not too short, not too long.
""".strip()

In [12]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [13]:
import json

user_prompt = json.dumps(doc)

In [14]:
messages = [
    {"role": "developer", "content": data_gen_instructions},
    {"role": "user", "content": user_prompt}
]

In [15]:
response = openai_client.responses.parse(
    model="gpt-5.4-mini",
    input=messages,
    text_format=Questions
)

In [16]:
result = response.output_parsed

print(result)

questions=['Can I still join the course if I just found out about it?', 'If I join late, can I still get a certificate somehow?', 'Do I need to submit the project before submissions close to get the certificate?', 'Is it okay to start the course after it has already begun?', 'What’s the deadline for project submission if I want the certificate?']


In [17]:
print(result.questions)

['Can I still join the course if I just found out about it?', 'If I join late, can I still get a certificate somehow?', 'Do I need to submit the project before submissions close to get the certificate?', 'Is it okay to start the course after it has already begun?', 'What’s the deadline for project submission if I want the certificate?']


In [20]:
from evaluation_utils import llm_structured

In [21]:
result, usage = llm_structured(
    openai_client,
    data_gen_instructions,
    user_prompt,
    Questions
)

print(result.questions)

['I just found this course late — can I still start it and join now?', 'Is it okay to enroll after the course has already begun?', 'If I join the course now, do I still have a chance to get a certificate?', 'What do I need to do to be eligible for the certificate if I’m starting late?', 'Can I still submit the project for certification even if I discovered the course after it started?']


In [22]:
from evaluation_utils import calc_price

In [23]:
cost = calc_price(usage)

cost

{'input_cost': 0.00015525, 'output_cost': 0.000441, 'total_cost': 0.00059625}

In [25]:
records = []

for q in result.questions:
    records.append({
        "question": q,
        "document": doc["doc_id"]
    })

records

[{'question': 'I just found this course late — can I still start it and join now?',
  'document': '74eb249bbf'},
 {'question': 'Is it okay to enroll after the course has already begun?',
  'document': '74eb249bbf'},
 {'question': 'If I join the course now, do I still have a chance to get a certificate?',
  'document': '74eb249bbf'},
 {'question': 'What do I need to do to be eligible for the certificate if I’m starting late?',
  'document': '74eb249bbf'},
 {'question': 'Can I still submit the project for certification even if I discovered the course after it started?',
  'document': '74eb249bbf'}]

llm_structured makes one structured-output call. llm_structured_retry wraps the same call in a retry loop. If one request fails because of a temporary API or network issue, it waits briefly and tries again.

In [26]:
from evaluation_utils import llm_structured_retry


The processing function takes one document and turns it into ground truth records.
For each document, we:

convert the document to JSON so we can send it to the LLM
ask the LLM to return a Questions object
create one ground truth record for each generated question

In [29]:
def generate_ground_truth(doc):
    user_prompt = json.dumps(doc)

    out, usage = llm_structured_retry(
        openai_client,
        data_gen_instructions,
        user_prompt,
        Questions
    )

    results = []

    for q in out.questions:
        results.append({
            "question": q,
            "document": doc["doc_id"]
        })

    return results, usage

<h3> Process one by one </h3>

In [30]:
from tqdm.auto import tqdm

ground_truth = []
usages = []

for doc in tqdm(documents[:5]):
    records, usage = generate_ground_truth(doc)
    ground_truth.extend(records)
    usages.append(usage)

  0%|          | 0/5 [00:00<?, ?it/s]

<h3> Parallel Processing </h3>

In [31]:
from concurrent.futures import ThreadPoolExecutor
from evaluation_utils import map_progress

In [32]:
with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, documents, generate_ground_truth)

  0%|          | 0/113 [00:00<?, ?it/s]

generate_ground_truth returns two things for each document: the generated records and the token usage.

In [33]:
ground_truth = []
usages = []

for records, usage in results:
    ground_truth.extend(records)
    usages.append(usage)

len(ground_truth)
# With 5 questions per document, you should get roughly 5x the number of documents.

565

In [34]:
from evaluation_utils import calc_price

total_cost = 0.0

for usage in usages:
    cost = calc_price(usage)
    total_cost = total_cost + cost["total_cost"]

total_cost

0.08824875000000001

In [35]:
from evaluation_utils import calc_total_price

calc_total_price(usages)

0.08824875000000001

In [36]:
import pandas as pd

df_ground_truth = pd.DataFrame(ground_truth)

In [38]:
df_ground_truth.to_csv("data/ground_truth-new.csv", index=False)